<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Vegetation Times series extraction class - Dev notebook

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("preprod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from VTS_service_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.extractors.lrts_functions import LrtsExtractor
extractor = LrtsExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
extractor.setup_lrts_parameters(
            start_date= "2021-01-01",
            end_date="2026-01-01",
            vegetation_index= "ndvi",
            extraction_mode="period",        # Options: period, specific_dates or windows
            target_dates=[], 
            historical_years=10,
            partial_frequency=50,
            kpi_filter={
            'kpi_name': 'Summer NDVI Accumulation',
            'aggregation': 'accumulation'
        }
)

extractor2 = LrtsExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
extractor2.setup_lrts_parameters(
            start_date= "2021-01-01",
            end_date="2026-01-01",
            vegetation_index= "NDVI",
            is_extrapolated=False,
            extraction_mode="windows",        # Options: period, specific_dates or windows
            target_dates=["2021-01-01","2026-01-01"], 
            historical_years=10,
            partial_frequency=50,
            kpi_filter={}
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS"
            }


#### Test get LRTS Data

In [ ]:
print("\n--- Test: get_lrts_api ---")
# Invoke the function
try:
    result = extractor.get_lrts_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")

In [ ]:
print("\n--- Test Extractor2: get_lrts_api ---")
# Invoke the function
try:
    result = extractor2.get_lrts_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")

#### Test safe wrapper

In [ ]:
print("\n--- Test: get_lrts_api ---")
# Invoke the function
try:
    safe_result = extractor2.get_lrts_api_safe(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")

#### Test response formatting

In [ ]:
print("\n--- Test: format_inseason_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_lrts_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_inseason_json: No valid data from API.")

#### Test filter_timeseries_kpi

In [ ]:
print(formatted_df.columns)

In [ ]:
# Add this import at the top of your notebook
from earthdaily.agriculture.core.api_utils import filter_timeseries_kpi, format_kpi_results

print("\n--- Test: filter_timeseries_kpi ---")

if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_lrts_json(safe_result["data"])
    if not formatted_df.empty:
        try:
            # Call the function directly (not as a method)
            kpi_result = filter_timeseries_kpi(
                timeseries_df=formatted_df,
                start_date='2025-06-01',
                end_date='2025-08-31',
                kpi_name='NDVI Accumulation',
                aggregation='accumulation',
                years=[2024, 2023, 2022],
                value_column='ndvi'
            )
            
            print("✅ KPI computation successful!")
            print(f"\n📊 KPI Summary:")
            print(f"  Current Value: {kpi_result['current_period']['value']}")
            print(f"  Historical Avg: {kpi_result['historical_avg']['value']}")
            print(f"  Difference: {kpi_result['comparison']['difference']}")
            print(f"  Change %: {kpi_result['comparison']['percent_change']}%")
            
            # Format results
            kpi_df = format_kpi_results(kpi_result)
            print(f"\n📋 Formatted KPI Results:")
            display(kpi_df)
            
        except Exception as e:
            print(f"❌ KPI computation failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️ Formatted DataFrame is empty")
else:
    print("⚠️ Skipping filter_timeseries_kpi: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "3a5yn53",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS",
    "start_date":"2025-06-01",
    "end_date":"2025-10-01",
    'years': [2024, 2023, 2022]
}).to_dict() 

result = extractor.process_single_entity_lrts(row)

print(result)

In [ ]:
result = extractor2.process_single_entity_lrts(row)
print(result)

### Test historical years validation (list, string, column_mapping)

Validates that `years` works as a native list, comma-separated string (pipeline flattening),
and via `column_mapping` remapping from `historical_seasons`.

In [ ]:
from earthdaily.agriculture.core.api_utils import validate_historical_years

# Test validate_historical_years directly
test_cases = [
    None,
    'ALL',
    [2024, 2023, 2022],
    '2024,2023,2022',
    5,
]

for tc in test_cases:
    result = validate_historical_years(tc)
    print(f'  {str(tc):30s} -> {result} ({type(result).__name__})')


### 🗺️ process_vegetation_TS_bulk_extraction_parallel

In [ ]:
from datetime import timedelta
import pandas as pd
top25 = manager.sfd_list.head(50)

# Rename column
top25 = top25.rename(columns={'sowingDate': 'start_date'})
top25=top25.rename(columns={"crop.id": "crop"})

# Convert start_date to datetime if not already
top25['start_date'] = pd.to_datetime(top25['start_date'])

# Add end_date as start_date + 50 days
top25['end_date'] = top25['start_date'] + pd.Timedelta(days=50)

# Convert back to string format (YYYY-MM-DD) if needed for your API
top25['start_date'] = top25['start_date'].dt.strftime('%Y-%m-%d')
top25['end_date'] = top25['end_date'].dt.strftime('%Y-%m-%d')
print(top25.columns)
# Launch extraction with 10 threads 

result = extractor.process_entity_lrts_single_bulk_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop",
    filter_value="CORN",
    filter_type="exclude" # filter type used to 'include' or 'exclude' row matching column and value filter
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
extractor.